In [1]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
import torch
import os


In [6]:
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATA_FILE = "spider_style_dataset.jsonl"  # your file
OUTPUT_DIR = "qwen_spider_full_sft_v2"              # NEW name

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# needed for gradient checkpointing later
model.config.use_cache = False


In [7]:
raw_ds = load_dataset("json", data_files=DATA_FILE, split="train")
len(raw_ds), raw_ds[0]


(2503,
 {'messages': [{'role': 'system', 'content': 'You are a text-to-SQL expert.'},
   {'role': 'user',
    'content': 'Given schema:\nCREATE TABLE journals (\n  journal_id INTEGER PRIMARY KEY,\n  name TEXT,\n  publisher TEXT,\n  issn TEXT,\n  impact_factor REAL\n);\n\nCREATE TABLE articles (\n  article_id INTEGER PRIMARY KEY,\n  title TEXT,\n  pub_date DATE,\n  journal_id INTEGER\n);\n\nCREATE TABLE authors (\n  author_id INTEGER PRIMARY KEY,\n  name TEXT,\n  affiliation TEXT,\n  country TEXT\n);\n\nCREATE TABLE article_authors (\n  aa_id INTEGER PRIMARY KEY,\n  article_id INTEGER,\n  author_id INTEGER\n);\n\nCREATE TABLE citations (\n  citation_id INTEGER PRIMARY KEY,\n  citing_article_id INTEGER,\n  cited_article_id INTEGER,\n  citation_year INTEGER\n);\n\nQuestion: What is the average impact factor of all journals?'},
   {'role': 'assistant',
    'content': 'SELECT AVG(impact_factor) FROM journals;'}]})

In [8]:
MAX_SEQ_LEN = 512

def build_prompt_and_answer(example):
    msgs = example["messages"]
    # assume last assistant is the answer
    assert msgs[-1]["role"] == "assistant"
    answer = msgs[-1]["content"]

    prompt_messages = msgs[:-1]  # system + user (+ any others before assistant)
    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,  # model should now generate the assistant
    )

    # ensure a clean separation
    full_text = prompt + answer

    # tokenize prompt and full_text separately
    tok_prompt = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )
    tok_full = tokenizer(
        full_text,
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )

    input_ids = tok_full["input_ids"]
    attention_mask = tok_full["attention_mask"]

    prompt_len = sum(tok_prompt["attention_mask"])  # number of real prompt tokens

    labels = [-100] * MAX_SEQ_LEN
    for i in range(prompt_len, MAX_SEQ_LEN):
        # only non-pad positions get labels
        if attention_mask[i] == 1:
            labels[i] = input_ids[i]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

proc_ds = raw_ds.map(
    build_prompt_and_answer,
    remove_columns=raw_ds.column_names,
    desc="Tokenizing & building labels",
)


Tokenizing & building labels:   0%|          | 0/2503 [00:00<?, ? examples/s]

In [9]:
proc_ds[0].keys(), len(proc_ds[0]["input_ids"]), len(proc_ds[0]["labels"])


(dict_keys(['input_ids', 'attention_mask', 'labels']), 512, 512)

In [10]:
BATCH_SIZE = 2          # per device
GRAD_ACCUM = 32         # 2 * 32 = 64 effective
EPOCHS = 1              # start with 1; you can try 2 later
LR = 5e-6

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=20,
    save_strategy="epoch",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    report_to="none",
)


In [11]:
data_collator = default_data_collator


In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=proc_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


/tmp/ipykernel_13504/1510094780.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
20,0.241200
40,0.183200


('qwen_spider_full_sft_v2/tokenizer_config.json',
 'qwen_spider_full_sft_v2/special_tokens_map.json',
 'qwen_spider_full_sft_v2/chat_template.jinja',
 'qwen_spider_full_sft_v2/vocab.json',
 'qwen_spider_full_sft_v2/merges.txt',
 'qwen_spider_full_sft_v2/added_tokens.json',
 'qwen_spider_full_sft_v2/tokenizer.json')

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

ft_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
if ft_tokenizer.pad_token is None:
    ft_tokenizer.pad_token = ft_tokenizer.eos_token

ft_model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
ft_model.eval()

def generate_from_ft(example, max_new_tokens=256):
    msgs = example["messages"]
    # build messages like at evaluation time: system+user only
    eval_msgs = [m for m in msgs if m["role"] != "assistant"]
    prompt = ft_tokenizer.apply_chat_template(
        eval_msgs,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = ft_tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        out_ids = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=ft_tokenizer.pad_token_id,
            eos_token_id=ft_tokenizer.eos_token_id,
        )

    gen_ids = out_ids[0, inputs["input_ids"].shape[1]:]
    return ft_tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

print(generate_from_ft(raw_ds[0]))


The tokenizer you are loading from 'qwen_spider_full_sft_v2' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SELECT AVG(impact_factor) FROM journals;
